# Day 1-02｜Roboflow Court Keypoint Dataset 與 YOLO Pose 微調
> Python 籃球運動資料分析課程

## 單元目標
- 確認球場 keypoint dataset 的資料夾格式與標註欄位。
- 必要時將 Roboflow COCO keypoint export 轉為 Ultralytics YOLO pose dataset。
- 依需求選擇以基礎權重或課程權重初始化，執行 YOLO pose 微調。
- 使用課程提供模型對參考影片輸出 court keypoint 推論 demo。


## 資料位置
- Roboflow COCO keypoint export：`assets/datasets/roboflow_court_coco/`
- 轉換後 YOLO pose dataset：`assets/datasets/roboflow_court_yolo_pose/`
- 課程提供 court keypoint 權重：`assets/models/court_keypoints/yolo26n_basketball_court_pose_best.pt`
- 影片 demo 來源：`assets/raw/reference_videos/`

## 操作流程
1. 檢查資料集與模型權重是否到位。
2. 視需要執行 COCO-to-YOLO Pose 轉換。
3. 以 `TRAIN_MODE` 控制是否做微調，以及初始化方式。
4. 使用模型對參考影片輸出 court keypoint demo video。


In [ ]:
from pathlib import Path
import sys

try:
    from google.colab import drive  # type: ignore[import-not-found]

    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

bootstrap_candidates = [
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
]
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，確認課程已同步到 /content/drive/MyDrive/basketball_hackathon/course。"
    )

from src.course_setup import DEFAULT_COURSE_ROOT, bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(DEFAULT_COURSE_ROOT, mount_drive=False)


## Roboflow Keypoint Export 格式
`_annotations.coco.json` 需包含 `categories[].keypoints`、`categories[].skeleton`，以及每筆 annotation 的 `keypoints` 欄位。

```text
assets/datasets/roboflow_court_coco/
├── train/
│   ├── _annotations.coco.json
│   └── *.jpg
├── valid/
│   ├── _annotations.coco.json
│   └── *.jpg
└── test/
    ├── _annotations.coco.json
    └── *.jpg
```

轉換完成後，YOLO pose dataset 會放在 `assets/datasets/roboflow_court_yolo_pose/`，其中 `data.yaml` 會帶入 `kpt_shape`、`flip_idx`、`keypoint_names` 與 `keypoint_skeleton`。


In [ ]:
from src.roboflow_utils import dataset_status

COURT_COCO_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_coco"
COURT_YOLO_POSE_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_yolo_pose"
COURT_MODEL_PATH = (
    COURSE_ROOT
    / "assets"
    / "models"
    / "court_keypoints"
    / "yolo26n_basketball_court_pose_best.pt"
)

print("COCO keypoint export:")
print(dataset_status(COURT_COCO_DIR))
print("\nYOLO pose dataset:")
print(dataset_status(COURT_YOLO_POSE_DIR))
print("\nmodel exists:", COURT_MODEL_PATH.exists(), COURT_MODEL_PATH)


In [ ]:
from src.roboflow_utils import convert_roboflow_coco_keypoints_to_yolo_pose, find_coco_annotation

RUN_CONVERSION = False

if RUN_CONVERSION:
    if find_coco_annotation(COURT_COCO_DIR / "train") is None:
        raise FileNotFoundError(
            "請先將 Roboflow COCO keypoint export 放到 assets/datasets/roboflow_court_coco/"
        )
    data_yaml = convert_roboflow_coco_keypoints_to_yolo_pose(
        COURT_COCO_DIR,
        COURT_YOLO_POSE_DIR,
        overwrite=True,
    )
    print("converted data.yaml:", data_yaml)
else:
    print("RUN_CONVERSION=False；若已放入 Roboflow COCO keypoint export，可改為 True 執行轉換。")


In [ ]:
TRAIN_MODE = "skip"
# 可選值：
# - "skip"   : 不訓練，直接使用課程提供模型
# - "base"   : 以 yolo26n-pose.pt 為初始化權重重新訓練
# - "course" : 以課程提供權重為初始化，繼續微調

if TRAIN_MODE in {"base", "course"}:
    from ultralytics import YOLO

    data_yaml = COURT_YOLO_POSE_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 YOLO pose data.yaml。請先完成 COCO-to-YOLO Pose 轉換。"
        )

    if TRAIN_MODE == "course":
        if not COURT_MODEL_PATH.exists():
            raise FileNotFoundError(
                "找不到課程提供的 court keypoint 權重，無法使用 course 微調模式。"
            )
        init_weights = COURT_MODEL_PATH
    else:
        init_weights = Path("yolo26n-pose.pt")

    model = YOLO(str(init_weights))
    results = model.train(
        data=str(data_yaml),
        epochs=120,
        imgsz=960,
        batch=-1,
        workers=8,
        patience=40,
        fliplr=0.0,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "court_keypoints"),
        name=f"yolo26n_basketball_court_pose_{TRAIN_MODE}",
        plots=True,
    )
    print("init weights:", init_weights)
    print(results)
else:
    print("TRAIN_MODE=skip；本單元直接使用 assets/models/court_keypoints/ 內的權重。")


In [ ]:
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from src.video_utils import display_video_in_notebook

from src.cv_utils import save_json, save_image_rgb, show_image
from src.geometry_utils import render_bev_court
from src.yolo_utils import (
    bev_court_bounds,
    court_keypoints_from_result,
    draw_court_keypoint_records,
    write_court_keypoint_preview_video,
)

BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"
INFERENCE_MODEL_PATH = COURT_MODEL_PATH
video_candidates = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if not video_candidates:
    raise FileNotFoundError("找不到參考影片。請確認 assets/raw/reference_videos/ 內至少有一支 mp4。")
video_path = video_candidates[0]

cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise FileNotFoundError(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 60)
ok, frame_bgr = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"無法讀取 frame 60: {video_path}")
frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
bev = render_bev_court(BEV_SPEC_PATH)

model = YOLO(str(INFERENCE_MODEL_PATH))
result = model.predict(frame, conf=0.15, imgsz=960, verbose=False)[0]
keypoints = court_keypoints_from_result(
    result,
    bev.shape,
    frame_index=60,
    anchor_confidence=0.25,
    court_bounds=bev_court_bounds(BEV_SPEC_PATH),
)

src = np.asarray([kp.image_xy for kp in keypoints], dtype=np.float32)
dst = np.asarray([kp.bev_xy for kp in keypoints], dtype=np.float32)
H = None
if len(src) >= 6:
    H, mask = cv2.findHomography(src, dst, method=cv2.RANSAC, ransacReprojThreshold=5.0)
    if H is not None and mask is not None and int(mask.ravel().sum()) < 6:
        H = None

frame_vis = draw_court_keypoint_records(frame, keypoints)
show_image(frame_vis, "Court keypoint inference on reference frame")

out_image = COURSE_ROOT / "assets" / "results" / "d1_02_court_keypoints.png"
out_json = COURSE_ROOT / "assets" / "results" / "d1_02_court_keypoints.json"
save_image_rgb(out_image, frame_vis)
save_json([record.__dict__ for record in keypoints], out_json)

print("video:", video_path)
print("keypoint count:", len(keypoints))
print("homography estimated:", H is not None)
print("saved frame preview:", out_image)
pd.DataFrame([record.__dict__ for record in keypoints]).head()

preview_video_path = COURSE_ROOT / "assets" / "results" / "d1_02_court_pose_demo.mp4"
preview_video_path, preview_rows = write_court_keypoint_preview_video(
    video_path=video_path,
    model_path=INFERENCE_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=preview_video_path,
    max_frames=120,
    conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=40,
)
preview_json = preview_video_path.with_suffix('.json')

print("saved video demo:", preview_video_path)
print("saved json:", preview_json)
print("frames:", len(preview_rows))
display_video_in_notebook(preview_video_path, loop=True)
